The Problem: KV Cache Alone Hits a Wall

With KV cache, you're still doing full attention for each new token — just caching K/V so you don't recompute them. But attention cost is O(seq_len × d_k) per token, and later tokens have to attend over an ever-growing context.

- Token 1:  Q₁ × [K₁]           → 1 dot product
- Token 2:  Q₂ × [K₁, K₂]       → 2 dot products  
- Token 3:  Q₃ × [K₁, K₂, K₃]   → 3 dot products
- ...
- Token 32k: Q₃₂ₖ × [K₁, ..., K₃₂ₖ] → 32k dot products

Memory/compute explodes linearly with sequence length.

---

Ring Attention: Parallelize the Sequence Dimension, split the sequence across N GPUs. Each GPU holds:

- GPU 0: Q tokens [0, seq/N)     K/V tokens [0, seq/N)
- GPU 1: Q tokens [seq/N, 2seq/N) K/V tokens [seq/N, 2seq/N)
- ...

But here's the trick: instead of each GPU computing full attention (impossible — it only has local K/V), GPUs pass K/V around a ring:
- Step 0: GPU 0 has K/V chunk 0, computes partial attention
- Step 1: GPU 0 receives K/V chunk 1, computes next partial
- Step 2: GPU 0 receives K/V chunk 2, computes next partial
- ...
- Step N: GPU 0 has now attended over ALL K/V chunks

Each GPU's Q never moves — only K/V travel through the ring.

There are more efficient methods, like zig zag attention, but let us try to understand native ring attention first.

## Sequence Splitting at Data Loading

Each CP rank gets a contiguous chunk of the sequence, sliced during the data loading/collate phase. 

In [16]:
import torch
import torch.nn.functional as F

# A tiny sequence: 4 tokens, each represented by a simple embedding
seq_length = 4
vocab_size = 100
embed_dim = 4

# "Tokens" as simple one-hot-ish vectors for visibility
# Token 0, 1, 2, 3 (in practice these would be word IDs)
token_ids = torch.tensor([10, 20, 30, 40])  # just IDs for display
global_positions = torch.arange(seq_length)
print(f"Full sequence: {token_ids.tolist()}")
print(f"Global positions: {global_positions.tolist()}")

Full sequence: [10, 20, 30, 40]
Global positions: [0, 1, 2, 3]


In [17]:
# Simulate 2 CP ranks
cp_size = 2
seq_per_rank = seq_length // cp_size  # 2 tokens per rank


print(f"\nSplit across {cp_size} ranks, {seq_per_rank} tokens each:")

for cp_rank in range(cp_size):

    start = cp_rank * seq_per_rank
    end = start + seq_per_rank

    tokens = token_ids[start:end]
    positions = global_positions[start:end]

    print(f"  Rank {cp_rank}: tokens={tokens.tolist()}, positions={positions.tolist()}")


Split across 2 ranks, 2 tokens each:
  Rank 0: tokens=[10, 20], positions=[0, 1]
  Rank 1: tokens=[30, 40], positions=[2, 3]


## Make Q K V

Now, assume we divided the tokens across 2 different ranks, we can create our q, k, v vectors.

Again... this is just for demonstration

In [18]:
def make_dummy_qkv(token_slice, n_heads=1, dim=embed_dim):

    q = torch.rand(len(token_slice), dim)
    k = torch.rand(len(token_slice), dim)
    v = torch.rand(len(token_slice), dim)
    
    return q, k, v


In [19]:
n_heads = 1
q_ranks = []
k_ranks = []
v_ranks = []


for cp_rank in range(cp_size):

    start = cp_rank * seq_per_rank
    end = start + seq_per_rank
    local_tokens = token_ids[start:end]

    q, k, v = make_dummy_qkv(local_tokens, n_heads)
    q_ranks.append(q)
    k_ranks.append(k)
    v_ranks.append(v)

    print()
    print(f"Rank {cp_rank}:")
    print(f"  Q shape: {q.shape}, K shape: {k.shape}, V shape: {v.shape}")
    print(f"  Q:\n{q.squeeze(0)}\n  K:\n{k.squeeze(0)}\n  V:\n{v.squeeze(0)}")


Rank 0:
  Q shape: torch.Size([2, 4]), K shape: torch.Size([2, 4]), V shape: torch.Size([2, 4])
  Q:
tensor([[0.2531, 0.3561, 0.5293, 0.8066],
        [0.9756, 0.9413, 0.6563, 0.5839]])
  K:
tensor([[0.8077, 0.6458, 0.7575, 0.3824],
        [0.3332, 0.1972, 0.9841, 0.1109]])
  V:
tensor([[0.5636, 0.4148, 0.5222, 0.8706],
        [0.5569, 0.5554, 0.1359, 0.6164]])

Rank 1:
  Q shape: torch.Size([2, 4]), K shape: torch.Size([2, 4]), V shape: torch.Size([2, 4])
  Q:
tensor([[0.6640, 0.7071, 0.2710, 0.4990],
        [0.0663, 0.0278, 0.4063, 0.8983]])
  K:
tensor([[0.0298, 0.6373, 0.0236, 0.0105],
        [0.3770, 0.4796, 0.3297, 0.7737]])
  V:
tensor([[0.6119, 0.1553, 0.7274, 0.7979],
        [0.5326, 0.1469, 0.8180, 0.6405]])


## Ring Attention

Now it's time to see how ring attention could be used here

In [22]:
def simple_attention_with_causal(q, k, v, q_positions, k_positions):

    d_k = q.shape[-1]
    sm_scale = 1.0 / (d_k ** 0.5)
    
     # Create causal mask based on global positions
    seq_q = q.shape[0]
    seq_k = k.shape[0]
    mask = k_positions.unsqueeze(0) <= q_positions.unsqueeze(1)  # (seq_q, seq_k)
    
    scores = torch.matmul(q, k.T) * sm_scale
    scores = scores.masked_fill(~mask, float('-inf'))
    
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, v)

    return output, attn_weights

Bruh I'm not sure about this part

## Picotron Full Implementation

Let's walk through Picotron's ring attention implementation step by step:

### Step 1: Initialization

Each GPU (rank) initializes with its local Q, K, V tensors. The key insight is that Q stays on its home GPU while K and V are passed around the ring.

```python
# From context_parallel.py - RingAttentionFunc.forward()
k_og = k.clone()  # Save original K/V for backward pass
v_og = v.clone()
out, lse = None, None  # Accumulator for attention output and log-sum-exp
```

### Step 2: Ring Communication Pattern

The `ContextCommunicate` class handles the ring passing of K/V tensors:

- **send_rank**: The next GPU in the ring to send to
- **recv_rank**: The previous GPU in the ring to receive from
- **P2P operations**: Uses `dist.batch_isend_irecv` for efficient overlap

```python
# From cp_communications.py
send_operation = dist.P2POp(dist.isend, tensor_to_send, self.send_rank)
recv_operation = dist.P2POp(dist.irecv, result_tensor, self.recv_rank)
self._pending_operations.extend([send_operation, recv_operation])
```

### Step 3: Forward Pass - Iterative Attention

For each step in the ring (total `world_size` steps):

1. **Send K/V to next rank, receive K/V from previous rank** (async, non-blocking)
2. **Compute partial attention** with current K/V chunk
3. **Merge using online softmax** to combine partial results
4. **Wait for communication to complete** before next iteration

```python
for step in range(comm.world_size):
    # Non-blocking send/recv of K,V to neighbors
    if step + 1 != comm.world_size:
        next_k = comm.send_recv(k)
        next_v = comm.send_recv(v)
        comm.commit()

    # Compute attention for this K,V chunk
    if not is_causal or step <= comm.rank:
        block_out, block_lse = ring_attention_forward(
            q, k, v, sm_scale, is_causal and step == 0
        )
        # Online softmax merge: out = merge(out, block_out)
        out, lse = update_out_and_lse(out, lse, block_out, block_lse)

    # Wait and rotate K,V for next iteration
    if step + 1 != comm.world_size:
        comm.wait()
        k = next_k
        v = next_v
```

### Step 4: Online Softmax Trick

The `update_out_and_lse` function merges attention outputs numerically stably:

```python
# Numerically stable online softmax merge
current_out = current_out - F.sigmoid(block_lse - current_lse) * (current_out - block_out)
current_lse = current_lse - F.logsigmoid(current_lse - block_lse)
```

This computes: `out = exp(lse) * out + exp(block_lse) * block_out` in a stable way.

### Step 5: Backward Pass

The backward pass is more complex because we need to:

1. Reverse the ring direction for gradient flow
2. Compute dQ locally, but dK/dV need to be accumulated across ranks
3. Two separate rings: one for forward K/V, one for backward dK/dV gradients

```python
# Two communication rings in backward:
# - kv_comm: rotates original K,V (needed to recompute attention)
# - d_kv_comm: rotates dK,dV gradients backward

for step in range(kv_comm.world_size):
    # Rotate K,V forward through the ring
    next_k = kv_comm.send_recv(k)
    next_v = kv_comm.send_recv(v)

    # Compute local gradients
    block_dq, block_dk, block_dv = ring_attention_backward(...)

    # Accumulate dK,dV and rotate gradients backward
    dk = block_dk + next_dk
    dv = block_dv + next_dv
    next_dk = d_kv_comm.send_recv(dk)
```

### Step 6: RoPE for Context Parallel

When using RoPE (Rotary Position Embeddings), each rank must compute only its portion:

```python
def update_rope_for_context_parallel(cos, sin):
    size_per_partition = seq_len // cp_world_size
    start_idx = cp_rank * size_per_partition
    end_idx = (cp_rank + 1) * size_per_partition
    return cos[start_idx:end_idx], sin[start_idx:end_idx]
```

## Picotron Full Implementation

In [ ]:
import torch
import torch.nn.functional as F

# Simulate 2 context-parallel ranks
cp_size = 2

def context_parallel_ring_attention_forward(q, k, v, cp_rank, cp_world_size, sm_scale=1.0, is_causal=True):
    """
    Simulates ring attention forward pass for a single rank.
    Each rank holds local Q and receives K/V chunks from the ring.
    """
    out = None
    lse = None
    
    # Local K,V for this rank
    local_k = k.clone()
    local_v = v.clone()
    
    for step in range(cp_world_size):
        # At step 0, every rank has its own local K,V
        # At step 1, each rank receives K,V from neighbor
        # This simulates the ring rotation
        
        if not is_causal or step <= cp_rank:
            # Compute attention between local Q and current K,V chunk
            block_out, block_lse = ring_attention_forward_step(
                q, local_k, local_v, sm_scale, is_causal and step == 0
            )
            
            # Merge using online softmax
            if out is None:
                out = block_out
                lse = block_lse
            else:
                out, lse = merge_attention_outputs(out, lse, block_out, block_lse)
        
        # Simulate K,V rotation (in real impl, this is P2P communication)
        if step < cp_world_size - 1:
            # Next rank's K,V (simulated by swapping in this toy example)
            # Real impl: receive from recv_rank, send to send_rank
            pass
    
    return out, lse

def ring_attention_forward_step(q, k, v, sm_scale, is_causal):
    """Single step of ring attention - computes attention for one K,V chunk."""
    d_k = q.shape[-1]
    scores = torch.matmul(q, k.T) * sm_scale
    
    if is_causal:
        seq_len = scores.shape[-1]
        causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=q.device, dtype=torch.bool), diagonal=1)
        scores.masked_fill_(causal_mask, float('-inf'))
    
    # Online softmax
    S_max = torch.max(scores, dim=-1, keepdim=True)[0]
    exp_S = torch.exp(scores - S_max)
    exp_sum = torch.sum(exp_S, dim=-1, keepdim=True)
    log_sum_exp = torch.log(exp_sum) + S_max.squeeze(-1)
    
    attn_weights = exp_S / exp_sum
    out = torch.matmul(attn_weights, v)
    
    return out, log_sum_exp.squeeze(-1)

def merge_attention_outputs(out, lse, block_out, block_lse):
    """Numerically stable merge of two attention outputs using online softmax."""
    block_out = block_out.to(torch.float32)
    block_lse = block_lse.unsqueeze(-1)
    out = out - F.sigmoid(block_lse - lse.unsqueeze(-1)) * (out - block_out)
    lse = lse - F.logsigmoid(lse.unsqueeze(-1) - block_lse).squeeze(-1)
    return out, lse

# Test with toy example
seq_len = 4
embed_dim = 4
n_heads = 1
sm_scale = 1.0 / (embed_dim ** 0.5)

# Each "rank" gets 2 tokens
q_rank0 = torch.rand(2, embed_dim)
k_rank0 = torch.rand(2, embed_dim)
v_rank0 = torch.rand(2, embed_dim)

q_rank1 = torch.rand(2, embed_dim)
k_rank1 = torch.rand(2, embed_dim)
v_rank1 = torch.rand(2, embed_dim)

# Rank 0: attend over K,V from both ranks
out_rank0, lse_rank0 = context_parallel_ring_attention_forward(
    q_rank0, k_rank0, v_rank0, cp_rank=0, cp_world_size=2, sm_scale=sm_scale, is_causal=True
)

# Rank 1: attend over K,V from both ranks
out_rank1, lse_rank1 = context_parallel_ring_attention_forward(
    q_rank1, k_rank1, v_rank1, cp_rank=1, cp_world_size=2, sm_scale=sm_scale, is_causal=True
)

print(f"Rank 0 output shape: {out_rank0.shape}")
print(f"Rank 1 output shape: {out_rank1.shape}")

## Zig Zag Attention

Not Done